In [1]:
input = [['HEADER','MNC'],['A1','A11'],['A1','A11'],['A1','A11'],['A1','A12'],['K','K1'],['MNC','A'],['MNC','B'],['MNC','C'],['A','A1'],['A','A2'],['B','B1'],['C','C1'],['C','C2'],['B1','B11'],['B1','B12'],['B1','B13'],['B11','B111'],['B11','B112'],['B11','B113'],['C1','C11'],['C1','C12']]

In [2]:
import pandas as pd
import numpy as np

class sap_group_hierarchy:
  """
     Author: Sandipan Kar
     Date: 15/05/2023
     Changed By:
     Changed On:
     This module is designed to transform the group data into a format suitable for ingestion
     into the BAPI_COSTCENTERGROUP_CREATE/BAPI_PROFITCENTERGROUP_CREATE function module.
     The input data should be in a specific format with columns PARENT & CHILD(i.e. CC/PC Group).
  """
  hierarchy_list = []
  parent = []
  child = {}
  df_input = []
  hierarchy_list_display = []

  def __init__(self, df_input=[['PARENT','CHILD']]):
    self.df_input = pd.DataFrame(df_input, columns=['PARENT','CHILD'])

  def read_xls(self, path='hierarchy_input.xlsx'):
    self.df_input = pd.read_excel(path)
    self.df_input.columns = ['PARENT','CHILD']

  def get_input(self):
    pass

  def drop_duplicates(self):
    self.df_input = self.df_input.drop_duplicates()
    self.df_input = self.df_input.reset_index(drop=True)

  def groupInput(self):
    self.df_input.columns = ['PARENT','CHILD']
    parent = list(set(self.df_input['PARENT']))
    child = {}
    for i in range(0,len(parent)):
      nlist = []
      for j in range(0,len(self.df_input)):
        if(parent[i] == self.df_input['PARENT'][j]):
          nlist.append(self.df_input['CHILD'][j])
      child[parent[i]] = nlist
    self.parent = parent
    self.child = child

  def grouping_hierarchy(self, p, c, lvl):
    c_count = 0
    if(p == self.parent.index(self.child['HEADER'][0])):
      self.hierarchy_list = []
      self.hierarchy_list.append([c,lvl])
    lvl = lvl + 1
    while(c_count<len(self.child[c])):
      try:
        self.hierarchy_list.append([self.child[c][c_count],lvl])
        self.grouping_hierarchy(self.parent.index(self.child[c][c_count]), self.child[c][c_count], lvl)
      except ValueError:
        pass
      c_count=c_count+1

  def hierarchy_display(self, resolution=1):
    self.hierarchy_list_display = []
    for i in self.hierarchy_list:
      print('-'*i[1]*resolution,'→  ',i[0],sep='')
      str = '-'*i[1]*resolution+'→  '+i[0]
      self.hierarchy_list_display.append([str,i[1]])

  def return_hierarchy_list(self):
    return pd.DataFrame(self.hierarchy_list, columns=['GROUP', 'LEVEL'])

  def save_xls(self, path='hierarchy_output.xlsx'):
    xlsDf = pd.DataFrame(self.hierarchy_list, columns=['GROUP','LEVEL'])
    xlsDf.to_excel(path, index=False)

  def save_hierarchy_display(self, path='hierarchy_display_output.xlsx'):
    xlsDf = pd.DataFrame(self.hierarchy_list_display, columns=['Hierarchy Structure', 'Level'])
    xlsDf.to_excel(path, index=False)

  def create_hierarchy(self):
    self.drop_duplicates()
    self.groupInput()
    self.grouping_hierarchy(self.parent.index(self.child['HEADER'][0]), self.child['HEADER'][0], 0)

In [3]:
cc_grp = sap_group_hierarchy(input)

In [4]:
cc_grp.create_hierarchy()

In [5]:
cc_grp.return_hierarchy_list()

,GROUP,LEVEL
0,MNC,0
1,A,1
2,A1,2
3,A11,3
4,A12,3
5,A2,2
6,B,1
7,B1,2
8,B11,3
9,B111,4


In [6]:
cc_grp.hierarchy_display(3)

→  MNC
---→  A
------→  A1
---------→  A11
---------→  A12
------→  A2
---→  B
------→  B1
---------→  B11
------------→  B111
------------→  B112
------------→  B113
---------→  B12
---------→  B13
---→  C
------→  C1
---------→  C11
---------→  C12
------→  C2


In [7]:
cc_grp.df_input

,PARENT,CHILD
0,HEADER,MNC
1,A1,A11
2,A1,A12
3,K,K1
4,MNC,A
5,MNC,B
6,MNC,C
7,A,A1
8,A,A2
9,B,B1
